# Stage 4 — Walk-Forward Robustness

Apply the **chosen** configuration (from Stage 3) on a 6-month rolling window across the full dataset.

**Acceptance criterion:** ≥ 80% of test months must produce PF > 1.0.

This stage detects regime overfitting that a static IS/OOS split cannot catch. We are NOT re-optimizing per window — we use the single locked configuration.

In [ ]:
from __future__ import annotations

import json
from dataclasses import asdict
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from quant.data.loader import load_ohlcv_parquet
from quant.strategies.mean_reversion import MeanReversionConfig
from quant.validation.walk_forward import run_walk_forward

DATA_RAW = Path('../data/raw')
DATA_PROC = Path('../data/processed')
RESULTS = Path('../results')
SYMBOL = 'BTCUSDT'
INTERVAL = '1h'
THRESHOLD_PCT = 0.80

In [ ]:
df = load_ohlcv_parquet(DATA_RAW / f'{SYMBOL}_{INTERVAL}.parquet')
chosen = json.loads((DATA_PROC / 'chosen_config.json').read_text())
params = chosen['params']

base = MeanReversionConfig(name='mean_reversion', timeframe='1h')
config = MeanReversionConfig(**{**asdict(base), **params})

print('Walk-forward on chosen config:')
print(json.dumps(params, indent=2))
print(f'\nFull data: {len(df):,} bars  [{df.index.min()}  →  {df.index.max()}]')

In [ ]:
result = run_walk_forward(
    ohlcv=df,
    config=config,
    train_months=6,
    test_months=1,
    step_months=1,
    threshold_pf=1.0,
    threshold_pct=THRESHOLD_PCT,
)

print(f'Windows:        {len(result.windows)}')
print(f'% profitable:   {result.pct_profitable:.1%}  (threshold = {THRESHOLD_PCT:.0%})')
print(f'Passes Stage 4: {"YES" if result.passes else "NO"}')

In [ ]:
per_window = result.per_window_metrics.copy()
per_window['profitable'] = per_window['pf'] > 1.0
per_window[['window_start', 'window_end', 'n_trades', 'pf', 'sharpe', 'max_dd', 'profitable']].round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
colors = ['#10b981' if p else '#ef4444' for p in per_window['profitable']]
ax.bar(per_window['window_start'], per_window['pf'], color=colors, width=20)
ax.axhline(1.0, color='black', linestyle='--', lw=1, alpha=0.6, label='PF = 1.0')
ax.set_xlabel('Test month start')
ax.set_ylabel('Profit Factor')
ax.set_title(f'Walk-forward — PF per test month  ({result.pct_profitable:.0%} profitable)')
ax.legend()
ax.grid(alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()

out = RESULTS / 'figures' / '05_walkforward_pf.png'
fig.savefig(out, dpi=120, bbox_inches='tight')
print(f'Wrote: {out}')
plt.show()

## Stage 4 verdict

Pass → proceed to [06_holdout.ipynb](06_holdout.ipynb).  
Fail → terminate strategy + write [08_postmortem.ipynb](08_postmortem.ipynb).